In [1]:
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
from pathlib import Path
import os
import zipfile
import requests
def download_data(source: str, destination: str, remove_file: bool = True):
    data_path = Path('data')
    image_path = data_path/destination
    target_file = data_path/Path(source).name
    with open(target_file, 'wb') as f:
        request = requests.get(source)
        print('downloading data from the web')
        f.write(request.content)
    with zipfile.ZipFile(target_file, 'r') as zip_ref:
        print('Unzipping file....')
        zip_ref.extractall(image_path)

    if remove_file:
        os.remove(target_file)
    return image_path


In [3]:
image_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                           destination="pizza_steak_sushi")
image_path

downloading data from the web
Unzipping file....


WindowsPath('data/pizza_steak_sushi')

In [4]:
train_dir = image_path / 'train'
test_dir = image_path / 'test'

In [5]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
IMG_SIZE = 224
manual_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor()
])
#here i wrote how to create a dataloader
def create_dataloaders(train_dir: str, test_dir: str, transforms: transforms.Compose, BATCH_SIZE : int = 32, NUM_WORKERS : int = os.cpu_count()):
    
    train_data = ImageFolder(train_dir, transform = transforms)
    test_data = ImageFolder(test_dir, transform = transforms)
    class_names = train_data.classes
    train_dataloader = DataLoader(
        train_data,
        batch_size = BATCH_SIZE,
        shuffle = True,
        num_workers = NUM_WORKERS
    )
    test_dataloader = DataLoader(
        test_data,
        batch_size = BATCH_SIZE,
        shuffle = False,
        num_workers = NUM_WORKERS
    )

    return train_dataloader, test_dataloader, class_names

train_dataloader, test_dataloader, class_names = create_dataloaders(train_dir = train_dir, test_dir = test_dir, transforms = manual_transforms)

In [6]:
train_dataloader

In [7]:
class_names

['pizza', 'steak', 'sushi']

In [8]:
img_batch, label_batch = next(iter(train_dataloader))
img_batch.shape

torch.Size([32, 3, 224, 224])

In [9]:
img, label = img_batch[0], label_batch[0]
img.shape

torch.Size([3, 224, 224])

In [10]:
height = 224
width = 224
channels = 3
patch_size = 16
num_patches = int(height * width / (patch_size **2))

patch_embedding_input = (height, width, channels)
patch_embedding_output = (num_patches, (patch_size**2) * channels)

In [11]:
patch_embedding_input, patch_embedding_output

((224, 224, 3), (196, 768))

In [24]:

assert IMG_SIZE // patch_size == 0, 'image size must be divisible by patch size'
fig, axes = plt.subplots(nrows = IMG_SIZE//patch_size, ncols = IMG_SIZE // patch_size, sharex=True, sharey = True, figsize = (num_patches, num_patches))
for i, patch_height in enumerate(range(0, IMG_SIZE, patch_size)):
    for j, patch_width in enumerate(range(0, IMG_SIZE, patch_size)):
        axes[i,j].imshow(img.permute(1,2,0)[patch_height: patch_height + patch_size, patch_width: patch_width + patch_size, :])
        axes[i,j].set_xlabel(j+1)
        axes[i,j].set_ylabel(i+1, rotation = 'horizontal', horizontalalignment = 'right', verticalalignment = 'center')
        axes[i,j].set_xticks([])
        axes[i,j].set_yticks([])
        axes[i,j].label_outer()

fig.suptitle(f'{class_names[label]} --> Pachified', fontsize = 15)
plt.show()

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [28]:
import torch
from torch import nn
Conv2d = nn.Conv2d(in_channels = 3, out_channels = 768, kernel_size = patch_size, stride = patch_size, padding = 0)

In [30]:
image_out_of_conv2d = Conv2d(img.unsqueeze(dim=0))
image_out_of_conv2d.shape

torch.Size([1, 768, 14, 14])

In [ ]:
fig, axes = plt.subplots(nrows = 1, ncols = 5, figsize = (12,12))
random_indexes = random.sample(range(0,768), k=5)
for i, idx in enumerate(random_indexes):
    axes[i].imshow(image_out_of_conv2d[:, idx,:,:].squeeze().detach().numpy())
    axes[i].set(xticklabels = [], yticklabels = [], xticks = [], yticks = [])

single_feature_image = image_out_of_conv2d[:,0,:,:]
single_features_image, single_features_image.requires_grad


In [32]:
flatten_layer = nn.Flatten(start_dim= 2, end_dim =3)
image_out_of_flatten_layer = flatten_layer(image_out_of_conv2d)
image_out_of_flatten_layer.shape

torch.Size([1, 768, 196])

In [36]:
print(f'normal image shape is {img.shape}')
print(f'image output shape after conv2d layer is {image_out_of_conv2d.shape}')
print(f'image output shape after the flatten layer is {image_out_of_flatten_layer.shape}')

normal image shape is torch.Size([3, 224, 224])
image output shape after conv2d layer is torch.Size([1, 768, 14, 14])
image output shape after the flatten layer is torch.Size([1, 768, 196])


In [40]:
image_permutted_flatten = image_out_of_flatten_layer.permute(0,2,1)
image_permutted_flatten.shape

torch.Size([1, 196, 768])

In [ ]:
single_flatten_layer = image_permutted_flatten[:,:,0]
plt.imshow(single_flatten_layer.detach().numpy())
plt.axis('Off')

In [62]:
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels: int= 3, patch_size: int= 16, embedding_dim: int= 768):
        super().__init__()
        self.Conv2d = nn.Conv2d(
            in_channels = in_channels,
            out_channels = embedding_dim,
            kernel_size = patch_size,
            stride = patch_size,
            padding = 0
        )
        self.flatten = nn.Flatten(start_dim = 2, end_dim = 3)
    def forward(self, x):
        IMG_RESOLUTION = x.shape[-1]
        assert IMG_RESOLUTION % patch_size == 0, 'image must be divisible by patch size'
        x = self.Conv2d(x)
        x = self.flatten(x)
        return x.permute(0,2,1)

In [64]:
pachify = PatchEmbedding()

In [66]:
image_out_pachify_layer = pachify(img.unsqueeze(0))
image_out_pachify_layer.shape

torch.Size([1, 196, 768])

In [80]:
batch_size = image_out_pachify_layer.shape[0]
embedding_dimension = image_out_pachify_layer.shape[-1]
class_embedding_token = nn.Parameter(torch.ones(1,1,embedding_dimension),requires_grad = True)
class_embedding_token.shape

torch.Size([1, 1, 768])

In [82]:
image_embedding_plus_class_token = torch.cat((image_out_pachify_layer, class_embedding_token), dim =1)
image_embedding_plus_class_token.shape

torch.Size([1, 197, 768])

In [84]:
position_embedding = nn.Parameter(torch.ones(batch_size, num_patches+1, embedding_dimension), requires_grad = True)
position_embedding.shape

torch.Size([1, 197, 768])

In [86]:
image_class_and_position_embedding=image_embedding_plus_class_token + position_embedding
image_class_and_position_embedding.shape

torch.Size([1, 197, 768])

In [88]:
class MultiheadSelfAttentionBlock(nn.Module):
    def __init__(self, embedding_dimension: int = 768, num_heads: int= 12, att_dropout : float = 0):
        super().__init__()
        self.layer_norm = nn.LayerNorm(normalized_shape = embedding_dimension)
        self.multiheadlayer = nn.MultiheadAttention(embed_dim = embedding_dimension, num_heads = num_heads, dropout = att_dropout, batch_first = True)

    def forward(self,x):
        x = self.layer_norm(x)
        x, _= self.multiheadlayer(query = x, key = x, value = x, need_weights = False)
        return x


In [90]:
multihead_self_attention_block = MultiheadSelfAttentionBlock()

In [92]:
image_out_multihead = multihead_self_attention_block(image_class_and_position_embedding)
image_out_multihead.shape

torch.Size([1, 197, 768])

In [98]:
class MultiLayerPerceptronBlock(nn.Module):
    def __init__(self, embedding_dim: int = 768, mlp_dropout: float = 0.1, mlp_size: int = 3072):
        super().__init__()
        self.layer_norm = nn.LayerNorm(normalized_shape = embedding_dim)
        self.mlp = nn.Sequential(
            nn.Linear(in_features = embedding_dim, out_features = mlp_size),
            nn.GELU(),
            nn.Dropout(p=mlp_dropout),
            nn.Linear(in_features = mlp_size, out_features = embedding_dim),
            nn.Dropout(p=mlp_dropout)
        )
    def forward(self, x):
        x = self.layer_norm(x)
        x = self.mlp(x)
        return x

In [100]:
mlp_block = MultiLayerPerceptronBlock()

In [102]:
image_out_of_mlp_block = mlp_block(image_out_multihead)
image_out_of_mlp_block.shape

torch.Size([1, 197, 768])

In [112]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, att_drop: float = 0, mlp_size :int= 3072, num_heads: int = 12, embedding_dim : int = 768, mlp_drop: float = 0.1):
        super().__init__()
        self.multiheadattention = MultiheadSelfAttentionBlock(embedding_dimension = embedding_dim, num_heads = num_heads, att_dropout = att_drop)
        self.multilayerperceptron = MultiLayerPerceptronBlock(embedding_dim = embedding_dim, mlp_dropout = mlp_drop, mlp_size = mlp_size)
    def forward(self, x):
        x = self.multiheadattention(x) + x
        x = self.multilayerperceptron(x) + x
        return x

In [114]:
transformer_encoder_block = TransformerEncoderBlock()
transformer_encoder_block

TransformerEncoderBlock(
  (multiheadattention): MultiheadSelfAttentionBlock(
    (layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (multiheadlayer): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
    )
  )
  (multilayerperceptron): MultiLayerPerceptronBlock(
    (layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (mlp): Sequential(
      (0): Linear(in_features=768, out_features=3072, bias=True)
      (1): GELU(approximate='none')
      (2): Dropout(p=0.1, inplace=False)
      (3): Linear(in_features=3072, out_features=768, bias=True)
      (4): Dropout(p=0.1, inplace=False)
    )
  )
)

In [118]:
loaded_transformer_encoder_block = nn.TransformerEncoderLayer(
    d_model = 768,
    nhead = 12,
    dim_feedforward = 3072,
    dropout = 0.1,
    activation = 'gelu',
    batch_first = True,
    norm_first = True,
)
loaded_transformer_encoder_block

TransformerEncoderLayer(
  (self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
  )
  (linear1): Linear(in_features=768, out_features=3072, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (linear2): Linear(in_features=3072, out_features=768, bias=True)
  (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (dropout1): Dropout(p=0.1, inplace=False)
  (dropout2): Dropout(p=0.1, inplace=False)
)

In [126]:
!pip install -q torchinfo
from torchinfo import summary
summary(transformer_encoder_block, input_size = (1,196,768),col_names=["input_size", "output_size", "num_params", "trainable"], col_width=20, row_settings=["var_names"])

Layer (type (var_name))                                 Input Shape          Output Shape         Param #              Trainable
TransformerEncoderBlock (TransformerEncoderBlock)       [1, 196, 768]        [1, 196, 768]        --                   True
├─MultiheadSelfAttentionBlock (multiheadattention)      [1, 196, 768]        [1, 196, 768]        --                   True
│    └─LayerNorm (layer_norm)                           [1, 196, 768]        [1, 196, 768]        1,536                True
│    └─MultiheadAttention (multiheadlayer)              --                   [1, 196, 768]        2,362,368            True
├─MultiLayerPerceptronBlock (multilayerperceptron)      [1, 196, 768]        [1, 196, 768]        --                   True
│    └─LayerNorm (layer_norm)                           [1, 196, 768]        [1, 196, 768]        1,536                True
│    └─Sequential (mlp)                                 [1, 196, 768]        [1, 196, 768]        --                   True
│  

In [128]:
summary(loaded_transformer_encoder_block, input_size = (1,196,768),col_names=["input_size", "output_size", "num_params", "trainable"], col_width=20, row_settings=["var_names"])

Layer (type (var_name))                            Input Shape          Output Shape         Param #              Trainable
TransformerEncoderLayer (TransformerEncoderLayer)  [1, 196, 768]        [1, 196, 768]        --                   True
├─LayerNorm (norm1)                                [1, 196, 768]        [1, 196, 768]        1,536                True
├─MultiheadAttention (self_attn)                   [1, 196, 768]        [1, 196, 768]        2,362,368            True
├─Dropout (dropout1)                               [1, 196, 768]        [1, 196, 768]        --                   --
├─LayerNorm (norm2)                                [1, 196, 768]        [1, 196, 768]        1,536                True
├─Linear (linear1)                                 [1, 196, 768]        [1, 196, 3072]       2,362,368            True
├─Dropout (dropout)                                [1, 196, 3072]       [1, 196, 3072]       --                   --
├─Linear (linear2)                             

In [ ]:
class vit(nn.Module):
    def __init__(self,in_channels = 3,embedding_dim: int =768, embedding_dropout: float = 0.1, num_classes: int = 1000, num_transformer_layers: int = 12, mlp_size: int = 3072,num_heads: int = 12, att_drop: float = 0, mlp_dropout: float = 0.1, img_size : int =224, patch_size : int = 16):
        super().__init__()
        assert img_size % patch_size == 0, 'image size must be divisible by patch size'
        self.num_patches = (img_size * img_size)//(patch_size**2)
        self.class_embedding = nn.Parameter(torch.randn(1, 1, embedding_dim), requires_grad = True)
        self.position_embedding = nn.Parameter(torch.randn(1,self.num_patches + 1, embedding_dim), requires_grad = True)
        self.embedding_dropout = nn.Dropout(p=embedding_dropout)
        self.patch_embedding= PatchEmbedding(in_channels = in_channels, patch_size = patch_size, embedding_dim = embedding_dim)
        self.TransformerEncoder =  nn.Sequential(*[TransformerEncoderBlock(embedding_dim=embedding_dim,
                                                                            num_heads=num_heads,
                                                                            mlp_size=mlp_size,
                                                                            mlp_drop=mlp_dropout) for _ in range(num_transformer_layers)])
        self.classifier = nn.Sequential(
            nn.LayerNorm(normalized_shape = embedding_dim),
            nn.Linear(in_features = embedding_dim, out_features = num_classes)
        )
    def forward(self, x):
        batch_size = x.shape[0]
        class_token = self.class_embedding.expand(batch_size, -1, -1)
        x = self.patch_embedding(x)
        x = torch.cat((class_token, x), dim = 1)
        x = self.position_embedding + x
        x = self.embedding_dropout(x)
        x = self.TransformerEncoder(x)
        x = self.classifier(x[:, 0])
        return x

In [168]:
torch.manual_seed(42)
random_image_tensor = torch.randn(1,3,224,224)
Vit = vit(num_classes = len(class_names))
Vit(random_image_tensor)

tensor([[0.2525, 0.8748, 1.2374]], grad_fn=<AddmmBackward0>)

In [172]:
summary(vit(), 
        input_size = (32,3,224,224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
       )

Layer (type (var_name))                                           Input Shape          Output Shape         Param #              Trainable
vit (vit)                                                         [32, 3, 224, 224]    [32, 1000]           152,064              True
├─PatchEmbedding (patch_embedding)                                [32, 3, 224, 224]    [32, 196, 768]       --                   True
│    └─Conv2d (Conv2d)                                            [32, 3, 224, 224]    [32, 768, 14, 14]    590,592              True
│    └─Flatten (flatten)                                          [32, 768, 14, 14]    [32, 768, 196]       --                   --
├─Dropout (embedding_dropout)                                     [32, 197, 768]       [32, 197, 768]       --                   --
├─Sequential (TransformerEncoder)                                 [32, 197, 768]       [32, 197, 768]       --                   True
│    └─TransformerEncoderBlock (0)                           

In [174]:
optimizer = torch.optim.Adam(
    params = vit().parameters(),
    lr = 3e-3,
    betas = (.9,.999),
    weight_decay = 0.3
)
loss_fn = nn.CrossEntropyLoss()

try:
    from torch.utils.tensorboard import SummaryWriter
except:
    !pip install -q tensorboard
    from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter()

In [192]:
from tqdm.auto import tqdm
def train_step(model: torch.nn.Module, dataloader: torch.utils.data.DataLoader, loss_fn: torch.nn.Module, optimizer: torch.optim.Optimizer ):
    model.train()
    train_loss, train_acc = 0, 0
    for batch, (X,y) in enumerate(dataloader):
        y_logits = model(X)
        y_prob = torch.softmax(y_logits, dim=1)
        y_pred = torch.argmax(y_prob, dim= 1)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_acc += (y_pred == y).sum().item()/len(y_pred)
    train_loss /= len(dataloader)
    train_acc /= len(dataloader)
    return train_loss, train_acc
def test_step(model: torch.nn.Module, dataloader: torch.utils.data.DataLoader, loss_fn: torch.nn.Module):
    model.eval()
    test_loss, test_acc = 0, 0
    with torch.inference_mode():
        for batch, (X,y) in enumerate(dataloader):
            test_logits = model(X)
            test_pred = test_logits.softmax(dim=1).argmax(dim=1)
            test_loss += loss_fn(test_pred, y).item()
            test_acc += (test_pred == y).sum().item()/len(test_pred)
        test_loss /= len(dataloader)
        test_acc /= len(dataloader)

    return test_loss, test_acc

def train(model: torch.nn.Module, train_dataloader: torch.utils.data.DataLoader, test_dataloader: torch.utils.data.DataLoader, epochs: int, optimizer: torch.optim.Optimizer, loss_fn: torch.nn.Module, writer: torch.utils.tensorboard.writer.SummaryWriter):
    results = {
        'train_loss': [],
        'train_acc': [],
        'test_loss': [],
        'test_acc': []
    }
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model = model, dataloader = train_dataloader, loss_fn = loss_fn, optimizer = optimizer)
        test_loss, test_acc = test_step(model = model, dataloader = test_dataloader, loss_fn = loss_fn)
        print(f'epochs {epoch+1}')
        print(f'train loss is {train_loss:.4f}')
        print(f'train accuracy is {train_acc:.4f}')
        print(f'test loss is {test_loss:.4f}')
        print(f'test accuracy is {test_acc:.4f}')

        results['train_loss'].append(train_loss)
        results['train_acc'].append(train_acc)
        results['test_loss'].append(test_loss)
        results['test_acc'].append(test_acc)

        writer.add_scalars(main_tag = 'Loss', tag_scalar_dict = {'train_loss': train_loss, 'test_loss': test_loss}, global_step = epoch)
        writer.add_scalars(main_tag = 'Accuracy', tag_scalar_dict = {'train_acc': test_acc, 'test_acc':test_acc}, global_step = epoch)
        writer.add_graph(model = model, input_to_model = torch.randn(32,3,224,224))
     writer.close()   

    return results

In [194]:
def set_seeds(seed: int =42):
    torch.manual_seed(42)

In [ ]:
set_seeds()
results = train(
    model=vit(),
    train_dataloader = train_dataloader,
    test_dataloader = test_dataloader,
    epochs = 5,
    loss_fn = loss_fn,
    optimizer = optimizer,
    writer = create_writer(experiment_name = 'data_20_percent', model_name = 'effnetb0', extra = f'{epoch} epochs')
)

  0%|          | 0/5 [00:00<?, ?it/s]

In [1]:
import torchvision
def plot_loss_curves(results):
    loss = results['train_loss']
    test_loss = results['test_loss']
    accuracy = results['train_acc']
    test_accuracy = results['test_acc']
    epochs = range(len(results['train_loss']))
    plt.subplots(1,2,1)
    plt.plot(epochs, loss, label = 'train_loss')
    plt.plot(epochs, test_loss, label = 'test_loss')
    plt.legend()
    plt.subplots(1,2,2)
    plt.plot(epochs, accuracy, label = 'train_acc')
    plt.plot(epochs, test_accuracy, label = 'test_acc')
    plt.legend()

pretrained_vit_weights = torchvision.models.ViT_B_16_Weights.DEFAULT
weights_transforms = pretrained_vit_weights.transforms()
pretrained_vit = torchvision.models.vit_b_16(weights = weights_transforms)



C:\Users\ABDALAH\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to C:\Users\ABDALAH/.cache\torch\hub\checkpoints\vit_b_16-c867db91.pth


URLError: <urlopen error [Errno 11001] getaddrinfo failed>

In [ ]:
for params in pretrained_vit.parameters():
    params.requires_grad = False

pretrained_vit.heads = nn.Linear(in_features = 768, out_features = len(class_names))

def save_model(model: torch.nn.Module, target_dir: str, model_name: str):
    model_path = Path(target_dir)
    model_path.mkdir(parents = True, exist_ok = True)
    assert model_name.endswith('.pt') or model_name.endswith('.pth'), 'model name should end with .pth or pt'
    model_save_path = model_path/model_name
    torch.save( obj= model.state_dict(), f = model_save_path)



In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

def create_writer(experiment_name: str, model_name: str, extra: str=None):
    from datetime import datetime
    import os
    timestamp = datetime.now().strftime(%Y-%m-%d)
    if extra:
        logdir = os.path.join('runs', experiment_name, model_name, timestamp, extra)
    else:
        logdir = os.path.join('runs',experiment_name, model_name, timestamp)
    print(f'Created SummaryWriter ...{logdir}')
    return SummaryWriter(logdir = logdir)

In [ ]:
data_dir = Path('data')
data_path = data_dir/'food-101'/'images'
amount_to_get = 0.2
def get_subset(
    image_path = data_path,
    datasplits = ['train', 'test'],
    classes = ['pizza', 'sushi', 'steak'],
    amount=0.1,
    seed=42
):
    label_splits = {}
    for datasplit in datasplits:
        print(f'creating folder file for {datasplit}.txt')
        label_path = data_dir / 'food-101'/'meta'/f'{datasplit}.txt'
        with open(label_path, 'r') as f:
            labels = [food.strip('\n') for food in f.readlines() if food.split('/')[0] in classes]

        number_to_sample = round(amount*len(labels))
        sampled_images = random.sample(labels, k = number_to_sample)
        image_paths = [pathlib.Path(str(image_path/sample_image) + '.jpg') for sample_image in sampled_images]
        label_splits[datasplit] = image_paths
    return label_splits

In [ ]:
def dataset(amount, data):
    len1 = amount* len(data)
    len2 = len(data) - len1
    random_split1, random_split2 = torch.utils.data.random_split(data, [len1, len2], generator = torch.manual_seed(seed))
    return random_split1, random_split2 